In [13]:
%reload_ext autoreload
%autoreload 2

import logging
logging.basicConfig(level=logging.INFO)
from rexpand_pyutils_file import write_file

from utils.conversation_data import load_conversation_data
from utils.json import to_json_compatible
from models.context import Context
from nodes.orchestrator import orchestrate
from models.workflow import State


In [14]:
# Load all conversations
conversations = load_conversation_data('./input/convo_2454_rows.xlsx')


INFO:root:Read sheet 'Result 1' from Excel file ./input/convo_2454_rows.xlsx
INFO:root:Read EXCEL data from ./input/convo_2454_rows.xlsx with 2454 rows


In [21]:
# Select a conversation to process
index = 9
messages_end = 3
messages = conversations[index][:messages_end]
context = Context(messages=messages)
write_file('./output/current_messages.json', to_json_compatible(messages))


INFO:root:Wrote JSON data to ./output/current_messages.json


In [22]:
# Fresh start. In this case, no human in the loop with topics suggested
state = State(context=context)
state = orchestrate(state)

print(state.step)
print(state.classified_category)
print(state.suggested_topics)
print(state.selected_topics)
print(state.generated_reply_message)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:root:Wrote JSON data to ./.cache/e90ab327d159dfe92ebc83e9d512f9a5.json
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 400 Bad Request"


BadRequestError: Error code: 400 - {'error': {'message': "Invalid schema for response_format 'question_detection_result': In context=(), 'additionalProperties' is required to be supplied and to be false.", 'type': 'invalid_request_error', 'param': 'text.format.schema', 'code': 'invalid_json_schema'}}

In [20]:
# Continue with selecting all suggested topics to generate a reply
state.selected_topics = state.suggested_topics
state = orchestrate(state)

print(state.step)
print(state.classified_category)
print(state.suggested_topics)
print(state.selected_topics)
print(state.generated_reply_message)


INFO:root:Read JSON data from ./.cache/14c6cc6adb02dd8d9c52f38e8c181de5.json


end: reply generated
ClassifierResult(category='no_reply', confidence=1.0, reason='The last message is from the job seeker asking to connect and discuss a role, but there is no reply from the referrer yet.', referenced_message_ids=['b99a542c-ccde-46a4-ace4-c570b14956ad'])
TopicSuggesterResult(topics=[Topic(topic='Follow-up', confidence=0.9, reason='The job seeker has sent an initial message to connect and discuss the role but has not received any reply yet. A polite follow-up would be appropriate to re-engage the referrer.'), Topic(topic='Express interest', confidence=0.8, reason='Reiterating interest in the Senior Associate role at VaynerMedia can help emphasize motivation and keep the conversation active.'), Topic(topic='Ask for a brief call', confidence=0.7, reason='Suggesting a brief call can facilitate a more direct and personal conversation about the role and potential referral.')])
TopicSuggesterResult(topics=[Topic(topic='Follow-up', confidence=0.9, reason='The job seeker has s